# LDM Training with Weights & Biases Tracking

This notebook demonstrates how to train a **Latent Diffusion Model** (LDM) with
real-time experiment tracking using [Weights & Biases](https://wandb.ai/).

**What's new compared to `06_ldm_training.ipynb`:**
- `WandbConfig` enables automatic scalar metric logging
- Metrics organized into three wandb sections: `train/`, `val/`, `viz/`
- `on_epoch_end` callback logs custom visualizations:
  - Individual broadband magnitude CDFs (V, g, r, z) — real vs generated
  - Individual airglow line CDFs (OI 5577, Na I D, OH, O2, etc.) with EMD
  - Latent CDF comparison (real vs generated latents)
  - Conditional validation grids (features vs transparency)
- `early_stop_on_ema=True` (default): early stopping uses EMA model validation loss
- Both base and EMA validation losses are tracked and logged
- `tqdm` progress bar with live train/val/ema_val/best metrics
- Optional: hyperparameter sweep example

**Requirements:**
```bash
pip install desisky[wandb,viz]
# or: pip install desisky[all]
```

## 1. Imports

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import equinox as eqx
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset
import wandb

from desisky.data import (
    SkySpecVAC,
    attach_solar_flux,
    add_galactic_coordinates,
    add_ecliptic_coordinates,
    compute_broadband_mags,
    measure_airglow_intensities,
    BROADBAND_NAMES,
    AIRGLOW_CDF_NAMES,
)
from desisky.io import load_builtin
from desisky.models.ldm import make_UNet1D_cond, compute_sigma_data
from desisky.training import (
    LatentDiffusionTrainer,
    LDMTrainingConfig,
    NumpyLoader,
    WandbConfig,
    fit_conditioning_scaler,
    normalize_conditioning,
)
from desisky.training.wandb_utils import log_figure, log_epoch_metrics
from desisky.inference import LatentDiffusionSampler
from desisky.visualization import (
    plot_cdf_comparison,
    plot_latent_corner_comparison,
    plot_conditional_validation_grid,
)

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.7.1
JAX devices: [CudaDevice(id=0), CudaDevice(id=1)]


## 2. wandb Login

In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.
wandb: Currently logged in as: mdowicz to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Load Dark-Time Data with Enrichment

In [3]:
vac = SkySpecVAC(version="v1.0", download=True, verify=True)
wavelength, flux, metadata = vac.load_dark_time(enrich=True)

metadata = attach_solar_flux(metadata, time_tolerance="12h")
metadata = add_galactic_coordinates(metadata)
metadata = add_ecliptic_coordinates(metadata)

print(f"Dark-time subset: {len(metadata)} exposures")
print(f"Flux shape: {flux.shape}")

Matched solar-flux values for 3362 / 3364 exposures (tolerance = 12h).
Dark-time subset: 3364 exposures
Flux shape: (3364, 7781)


## 4. Define Conditioning Columns

In [4]:
CONDITIONING_COLS = [
    "OBSALT", "TRANSPARENCY_GFA", "SUNALT", "SOLFLUX",
    "ECLLON", "ECLLAT", "GALLON", "GALLAT",
]

conditioning = metadata[CONDITIONING_COLS].to_numpy(dtype=np.float32)

finite_mask = np.isfinite(conditioning).all(axis=1)
n_bad = (~finite_mask).sum()
if n_bad > 0:
    print(f"Removing {n_bad} rows with NaN/Inf values")
    flux = flux[finite_mask]
    conditioning = conditioning[finite_mask]
    metadata = metadata.loc[finite_mask].reset_index(drop=True)

print(f"Conditioning shape: {conditioning.shape}")

Removing 2 rows with NaN/Inf values
Conditioning shape: (3362, 8)


## 5. Load VAE and Encode Spectra

In [5]:
vae, vae_meta = load_builtin("vae")

vae_inference = eqx.nn.inference_mode(vae)
results = vae_inference(flux, jr.PRNGKey(42))
latents = results["latent"]

print(f"Latent shape: {latents.shape}")

Latent shape: (3362, 8)


## 6. Create Data Loaders

In [6]:
latents_with_channel = np.array(latents[:, None, :])

dataset_size = len(latents_with_channel)
train_size = int(0.9 * dataset_size)

rng = np.random.default_rng(42)
indices = rng.permutation(dataset_size)
train_idx = indices[:train_size]
val_idx = indices[train_size:]

val_expids = metadata.iloc[val_idx]["EXPID"].tolist()

scaler_dict = fit_conditioning_scaler(conditioning[train_idx], CONDITIONING_COLS)
cond_train = normalize_conditioning(conditioning[train_idx], scaler_dict)
cond_val = normalize_conditioning(conditioning[val_idx], scaler_dict)

sigma_data = compute_sigma_data(latents_with_channel[train_idx])
print(f"sigma_data: {sigma_data:.6f}")

batch_size = 64
train_set = TensorDataset(
    torch.from_numpy(latents_with_channel[train_idx].astype(np.float32)),
    torch.from_numpy(cond_train.astype(np.float32)),
)
val_set = TensorDataset(
    torch.from_numpy(latents_with_channel[val_idx].astype(np.float32)),
    torch.from_numpy(cond_val.astype(np.float32)),
)

train_loader = NumpyLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = NumpyLoader(val_set, batch_size=batch_size, shuffle=False)

print(f"Training: {len(train_set):,} samples ({len(train_loader)} batches)")
print(f"Validation: {len(val_set):,} samples ({len(val_loader)} batches)")

sigma_data: 6.615777
Training: 3,025 samples (48 batches)
Validation: 337 samples (6 batches)


## 7. Initialize LDM Model

In [7]:
ldm = make_UNet1D_cond(
    in_ch=1, out_ch=1,
    meta_dim=len(CONDITIONING_COLS),
    hidden=32, levels=3, emb_dim=32,
    key=jr.PRNGKey(0),
)

print(f"LDM: meta_dim={len(CONDITIONING_COLS)}, hidden=32, levels=3")

LDM: meta_dim=8, hidden=32, levels=3


## 8. Configure Training + wandb

In [8]:
config = LDMTrainingConfig(
    epochs=100,
    learning_rate=1e-4,
    meta_dim=len(CONDITIONING_COLS),
    sigma_data=sigma_data,
    dropout_p=0.1,
    ema_decay=0.9999,
    early_stop_on_ema=True,
    validate_every=1,
    save_best=True,
    run_name="ldm_dark_wandb",
    val_expids=val_expids,
    conditioning_scaler=scaler_dict,
)

wconfig = WandbConfig(
    project="desisky-ldm",
    run_name=None,
    tags=["ldm", "dark-time", "edm", "ema"],
    log_every=1,
    viz_every=20,
)

## 9. Define Visualization Callback

The callback generates spectra using the EMA model, then produces:
1. **Broadband CDFs** — V, g, r, z magnitude comparison (real vs generated) with EMD
2. **Airglow CDFs** — one figure per emission line with EMD
3. **Latent corner plot** — pair-plot of real vs generated latent distributions with per-dimension EMD
4. **Conditional validation grid** — broadband magnitudes + airglow intensities vs SUNALT

In [9]:
wl_np = np.array(wavelength)
val_conditioning_raw = conditioning[val_idx]

# Index of SUNALT in CONDITIONING_COLS
SUNALT_IDX = CONDITIONING_COLS.index("SUNALT")


def on_epoch_end(model, ema_model, history, epoch):
    if epoch % wconfig.viz_every != 0:
        return
    if ema_model is None:
        return

    # --- Generate spectra with EMA model ---
    sampler = LatentDiffusionSampler(
        ldm_model=ema_model,
        vae_model=vae,
        sigma_data=config.sigma_data,
        conditioning_scaler=scaler_dict,
        num_steps=50,
        latent_channels=1,
        latent_dim=8,
    )

    n_gen = min(200, len(val_idx))
    gen_cond = jnp.array(val_conditioning_raw[:n_gen])
    generated = np.array(sampler.sample(
        key=jr.PRNGKey(epoch), conditioning=gen_cond, guidance_scale=1.0,
    ))
    real = np.array(flux[val_idx[:n_gen]])

    # --- Compute physical features once (reused for CDFs + cond grid) ---
    real_mags = compute_broadband_mags(wl_np, real)
    gen_mags = compute_broadband_mags(wl_np, generated)
    real_ag = measure_airglow_intensities(wl_np, real)
    gen_ag = measure_airglow_intensities(wl_np, generated)

    # --- 1. Broadband CDFs (one figure per filter: V, g, r, z) ---
    for j, band_name in enumerate(BROADBAND_NAMES):
        fig, _ = plot_cdf_comparison(
            real_mags[:, j:j+1], gen_mags[:, j:j+1], [band_name],
        )
        log_figure(f"viz/{band_name}", fig, epoch)
        plt.close(fig)

    # --- 2. Airglow CDFs (one figure per emission line) ---
    ag_names = [n for n in AIRGLOW_CDF_NAMES if n in real_ag.columns]
    for line_name in ag_names:
        fig, _ = plot_cdf_comparison(
            real_ag[[line_name]].to_numpy(),
            gen_ag[[line_name]].to_numpy(),
            [line_name],
        )
        safe_name = line_name.replace(" ", "_").replace("(", "-").replace(")", "-")
        log_figure(f"viz/{safe_name}", fig, epoch)
        plt.close(fig)

    # --- 3. Latent corner plot (real vs generated) ---
    real_latents = np.array(vae_inference(real, jr.PRNGKey(0))["latent"])
    gen_latents = np.array(vae_inference(generated, jr.PRNGKey(0))["latent"])

    fig_corner = plot_latent_corner_comparison(
        real_latents, gen_latents,
        labels=[f"z{i}" for i in range(real_latents.shape[1])],
    )
    log_figure("viz/latent_corner", fig_corner, epoch)
    plt.close(fig_corner)

    # --- 4. Conditional validation grid (physical features vs SUNALT) ---
    real_features = np.column_stack([real_mags, real_ag[ag_names].to_numpy()])
    gen_features = np.column_stack([gen_mags, gen_ag[ag_names].to_numpy()])
    feature_names = list(BROADBAND_NAMES) + ag_names

    fig_grid = plot_conditional_validation_grid(
        real_features, gen_features,
        cond_values=val_conditioning_raw[:n_gen, SUNALT_IDX],
        cond_name="SUNALT",
        feature_names=feature_names,
    )
    log_figure("viz/cond_grid_sunalt", fig_grid, epoch)
    plt.close(fig_grid)

print(f"Callback defined. Logs {len(BROADBAND_NAMES)} broadband CDFs"
      f" + {len(AIRGLOW_CDF_NAMES)} airglow CDFs"
      f" + latent corner + cond grid every {wconfig.viz_every} epochs.")

Callback defined. Logs 4 broadband CDFs + 6 airglow CDFs + latent corner + cond grid every 20 epochs.


## 10. Train with wandb Tracking

In [10]:
trainer = LatentDiffusionTrainer(
    ldm, config,
    wandb_config=wconfig,
    on_epoch_end=on_epoch_end,
)

print("Starting training...")
trained_ldm, ema_model, history = trainer.train(train_loader, val_loader)

print(f"Best val loss: {history.best_val_loss:.6f} (epoch {history.best_epoch})")

Starting training...


  wandb run: https://wandb.ai/mdowicz/desisky-ldm/runs/xxj262xp


LDM Training:   0%|          | 0/100 [01:07<?, ?it/s, train=0.9846, val=0.9802, ema_val=1.0950, best=1.0950]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training:  20%|██        | 20/100 [02:12<03:17,  2.46s/it, train=0.9033, val=0.9123, ema_val=1.1238, best=1.0610] /home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training:  40%|████      | 40/100 [03:06<02:27,  2.46s/it, train=0.8791, val=0.8667, ema_val=1.1232, best=1.0369]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training:  60%|██████    | 60/100 [04:01<01:40,  2.51s/it, train=0.8454, val=0.8528, ema_val=1.0428, best=1

  wandb run: https://wandb.ai/mdowicz/desisky-ldm/runs/xxj262xp


epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇██
train/loss,██▆▆▆▄▅▄▄▄▄▃▃▂▃▂▃▃▂▂▂▂▂▃▃▂▂▃▂▁▂▂▂▁▂▁▁▁▁▂
val/ema_loss,▅█▆▆▅▆▆▃▅▄█▄▅▅▆▄▅▃▅▇▃▆▃▃▅▁▆▅▂▅▄▄▃▄▃▆▃▆▃▁
val/loss,█▇█▅▆▄▇▅▃▄▄▄▅▃▅▅▃▁▃▃▄▄▂▃▁▁▃▁▂▂▂▃▃▂▁▄▃▂▂▃
epoch,99
train/loss,0.83554
val/ema_loss,1.04898
val/loss,0.84684


Best val loss: 1.012571 (epoch 94)


## 11. Summary

Your wandb dashboard now shows, organized into three sections:

**`train/` — Training scalars (every epoch):**
- `train/loss`

**`val/` — Validation scalars (every epoch):**
- `val/loss`, `val/ema_loss`

**`viz/` — Figures (every `viz_every` epochs):**
- `viz/V`, `viz/g`, `viz/r`, `viz/z` — individual broadband magnitude CDFs with EMD
- `viz/OI_5577`, `viz/Na_I_D`, `viz/OI_doublet`, `viz/OH`, `viz/O2-0-1-`, `viz/NI_5200` — individual airglow line CDFs with EMD
- `viz/latent_corner` — corner plot of real vs generated latent distributions with per-dimension EMD
- `viz/cond_grid_sunalt` — broadband magnitudes + airglow intensities vs SUNALT

## 12. Optional: Hyperparameter Sweep

In [11]:
sweep_config = {
    "method": "grid",
    "metric": {"name": "val/ema_loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"values": [5e-5, 3e-4]},
        "ema_decay": {"values": [0.999, 0.9999]},
        "dropout_p": {"value": 0.05},
    },
}

def train_sweep(config=None):
    # Must init before accessing wandb.config (sweep agent populates it)
    wandb.init()

    sweep_cfg = LDMTrainingConfig(
        epochs=50,
        learning_rate=wandb.config.learning_rate,
        meta_dim=len(CONDITIONING_COLS),
        sigma_data=sigma_data,
        dropout_p=wandb.config.dropout_p,
        ema_decay=wandb.config.ema_decay,
        early_stop_on_ema=True,
        validate_every=1,
        save_best=False,
        val_expids=val_expids,
        conditioning_scaler=scaler_dict,
    )

    sweep_wconfig = WandbConfig(
        project="desisky-ldm",
        tags=["sweep", "lr-ema-dropout"],
        log_every=1,
        viz_every=25,
    )

    sweep_model = make_UNet1D_cond(
        in_ch=1, out_ch=1, meta_dim=len(CONDITIONING_COLS),
        hidden=32, levels=3, emb_dim=32, key=jr.PRNGKey(0),
    )
    sweep_trainer = LatentDiffusionTrainer(
        sweep_model, sweep_cfg,
        wandb_config=sweep_wconfig,
        on_epoch_end=on_epoch_end,
    )
    sweep_trainer.train(train_loader, val_loader)

# Uncomment to run:
sweep_id = wandb.sweep(sweep_config, project="desisky-ldm")
wandb.agent(sweep_id, train_sweep, count=4)

Create sweep with ID: fnbewpvn
Sweep URL: https://wandb.ai/mdowicz/desisky-ldm/sweeps/fnbewpvn


wandb: Agent Starting Run: 8wjd8uz2 with config:
wandb: 	dropout_p: 0.05
wandb: 	ema_decay: 0.999
wandb: 	learning_rate: 5e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.


  wandb run: https://wandb.ai/mdowicz/desisky-ldm/runs/8wjd8uz2


LDM Training:   0%|          | 0/50 [00:40<?, ?it/s, train=0.9973, val=0.9881, ema_val=1.0876, best=1.0876]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training:  40%|████      | 20/50 [01:36<01:15,  2.51s/it, train=0.9263, val=0.9300, ema_val=1.0276, best=0.9746]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training: 100%|██████████| 50/50 [03:01<00:00,  3.63s/it, train=0.8770, val=0.8976, ema_val=0.9294, best=0.8884]

  wandb run: https://wandb.ai/mdowicz/desisky-ldm/runs/8wjd8uz2


...

wandb: Agent Starting Run: 70sy0jc7 with config:
wandb: 	dropout_p: 0.05
wandb: 	ema_decay: 0.9999
wandb: 	learning_rate: 0.0003
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.


  wandb run: https://wandb.ai/mdowicz/desisky-ldm/runs/70sy0jc7


LDM Training:   0%|          | 0/50 [00:39<?, ?it/s, train=0.9728, val=0.9680, ema_val=1.0948, best=1.0948]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training:  40%|████      | 20/50 [01:37<01:19,  2.67s/it, train=0.8876, val=0.8905, ema_val=1.1216, best=1.0592]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training:  80%|████████  | 40/50 [02:36<00:28,  2.86s/it, train=0.8796, val=0.8521, ema_val=1.1117, best=1.0326]/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
LDM Training: 100%|██████████| 50/50 [03:05<00:00,  3.70s/it, train=0.8514, val=0.8719, ema_val=1.1072, best=1.0326

  wandb run: https://wandb.ai/mdowicz/desisky-ldm/runs/70sy0jc7


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/loss,██▇█▆▆▅▅▄▄▅▄▄▄▃▄▄▃▃▄▃▃▃▂▂▃▃▃▃▂▂▂▂▂▂▁▂▂▁▂
val/ema_loss,▅▄▄▆▆▆▅▆▆▅▆▆▅█▃▆▆▇▃▄█▁▇▄▆▃▆▄▅▄▄▄▆▃▃▄▂▅▂▅
val/loss,████▇▇▄▆▅▅▆▆▆▆▅▄▇▅▃▄▄▄▅▄▅▃▄▅▄▅▁▃▃▃▄▃▅▄▃▄
epoch,49
train/loss,0.85136
val/ema_loss,1.10718
val/loss,0.87189
